In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
import re
from constants import KENPOM_FILE, TORVIK_FILE
from selenium import webdriver

In [3]:
def gen_browser():
    options = webdriver.FirefoxOptions()
    options.headless = True
    return webdriver.Firefox(options=options)

In [4]:
%%time
url = 'http://kenpom.com/index.php'

# Can also find archived version
# url = "https://web.archive.org/web/20220317101252/https://kenpom.com/"

# This doesn't work anymore
# f = requests.get(url)
# soup = BeautifulSoup(f.text, "lxml")

browser = gen_browser()
browser.get(url)
soup = BeautifulSoup(browser.page_source, "lxml")

CPU times: user 703 ms, sys: 35.1 ms, total: 738 ms
Wall time: 6.78 s


In [5]:
COLUMNS = [
    'Rank', 'Team', 'Conference', 'W-L', 'AdjustEM',
    'AdjustO', 'AdjustO Rank', 'AdjustD', 'AdjustD Rank',
    'AdjustT', 'AdjustT Rank', 'Luck', 'Luck Rank',
    'SOS Pyth', 'SOS Pyth Rank', 'SOS OppO', 'SOS OppO Rank',
    'SOS OppD', 'SOS OppD Rank', 'NCSOS Pyth', 'NCSOS Pyth Rank'
]

In [6]:
df = pd.DataFrame(
    [
        [tr.text for tr in row.find_all('td')]
        for row in soup.find('table', {'id': 'ratings-table'}).find_all(class_=r"tourney")
    ],
    columns=COLUMNS
)

In [7]:
df.shape

(64, 21)

In [8]:
# Returns true if given string is a number and a valid seed number (1-16)
def is_valid_seed(x):
    return str(x).replace(' ', '').isdigit() and int(x) > 0 and int(x) <= 16


def parse_name(row):
    if is_valid_seed(row['Team'][-2:]):
        row['Seed'] = row['Team'][-2:].strip()
        row['Team'] = row['Team'][:-2].strip()
    else:
        row['Seed'] = np.NaN
        row['Team'] = row['Team']
    return row


# Parse out seed/team
df = df.apply(parse_name, axis=1)

In [9]:
# Split W-L column into wins and losses
df['Wins'] = df['W-L'].apply(lambda x: int(re.sub('-.*', '', x)))
df['Losses'] = df['W-L'].apply(lambda x: int(re.sub('.*-', '', x)))
df.drop('W-L', inplace=True, axis=1)

# Reorder columns just cause I'm OCD
df = df[['Rank', 'Seed', 'Team', 'Conference', 'Wins', 'Losses', 'AdjustEM',
         'AdjustO', 'AdjustO Rank', 'AdjustD', 'AdjustD Rank',
         'AdjustT', 'AdjustT Rank', 'Luck', 'Luck Rank',
         'SOS Pyth', 'SOS Pyth Rank', 'SOS OppO', 'SOS OppO Rank',
         'SOS OppD', 'SOS OppD Rank', 'NCSOS Pyth', 'NCSOS Pyth Rank']]

# Drop non tournament teams
df = df.dropna()

In [10]:
df

,Rank,Seed,Team,Conference,Wins,Losses,AdjustEM,AdjustO,AdjustO Rank,AdjustD,...,Luck,Luck Rank,SOS Pyth,SOS Pyth Rank,SOS OppO,SOS OppO Rank,SOS OppD,SOS OppD Rank,NCSOS Pyth,NCSOS Pyth Rank
0,1,1,Duke,ACC,32,2,+38.85,127.9,4,89.1,...,+.049,61,+14.23,16,116.9,23,102.7,11,+10.06,18
1,2,1,Arizona,B12,32,2,+37.60,127.6,6,90.0,...,+.023,125,+14.91,9,117.6,15,102.7,8,+3.18,101
2,3,1,Michigan,B10,31,3,+37.58,126.6,8,89.0,...,+.045,66,+16.64,3,119.1,3,102.5,5,+12.46,11
3,4,1,Florida,SEC,26,7,+33.76,125.5,9,91.7,...,-.036,288,+15.98,4,119.3,2,103.3,25,+7.96,30
4,5,2,Houston,B12,28,6,+33.36,124.8,13,91.4,...,-.006,205,+13.52,24,116.8,25,103.3,24,+0.80,159
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,188,15,Furman,SC,22,12,-1.94,107.5,201,109.4,...,+.010,160,-6.25,291,108.1,215,114.3,356,-0.66,203
60,192,16,Siena,MAAC,23,11,-2.14,107.1,211,109.2,...,+.005,179,-9.52,348,102.8,351,112.4,300,-7.81,344
61,204,16,Howard,MEAC,24,10,-2.88,104.1,268,107.0,...,+.003,185,-14.01,364,100.8,363,114.8,360,-1.66,230
62,216,16,LIU,NEC,24,10,-3.96,105.6,239,109.6,...,+.104,12,-9.99,350,103.5,334,113.5,343,+2.10,121


In [12]:
df.to_csv(KENPOM_FILE.format(gender="mens"), index=False)

## Scrape Torvik

In [16]:
YEAR = 2026
url = f'https://barttorvik.com/ncaaw/trank.php?year={YEAR}&conlimit=NCAA'

In [17]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

browser = gen_browser()
browser.get(url)
element = WebDriverWait(browser, 10).until(
    EC.visibility_of_element_located((By.CLASS_NAME, "seedrow"))
)
soup = BeautifulSoup(browser.page_source, "lxml")

In [18]:
df = pd.DataFrame(
    [
        [tr.text for tr in row.find_all('td')]
        for row in soup.find_all(class_="seedrow")
    ],
).iloc[:, [1, 7]].rename(columns={7: 'pyth'})
df['Team'] = df[1].str.split("\xa0\xa0\xa0").apply(lambda x: x[0])
df['Seed'] = df[1].str.split("\xa0\xa0\xa0").apply(lambda x: x[1]).str.split(" seed").apply(lambda x: x[0])
df[['Team', 'Seed', 'pyth']].to_csv(TORVIK_FILE.format(gender='womens'), index=False)

# Old Version
This doesn't work with NIT seeds

In [ ]:
table_html = soup.find_all('table', {'id': 'ratings-table'})

In [3]:
# Weird issue w/ <thead> in the html
# Prevents us from just using pd.read_html
# Let's find all the thead contents and just replace/remove them
# This allows us to easily put the table row data into a dataframe using pandas
thead = table_html[0].find_all('thead')

table = table_html[0]
for x in thead:
    table = str(table).replace(str(x), '')

In [4]:
#    table = "<table id='ratings-table'>%s</table>" % table
df = pd.read_html(table, converters={3: lambda x: str(x)})[0]

df.columns = COLUMNS